In [100]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml
import os
import sys
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit

from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
             
import functions as fn
with open(project_root / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)


In [106]:
#----------------------------------------------------------------------------
# Defin the df: Use bitrate_dl as the target and test whether the variables already identified as explanatory can predict it.
#----------------------------------------------------------------------------

# 1. Define the dataset
#1.1 import qos file
qos_df = pd.read_csv(project_root/config["data"]["clean"]["file3"], sep=(","), dtype={"insee_com":"str"}, encoding = "latin1")
qos_df = qos_df[['measure_id','insee_com','operator','bitrate_dl','rsrp', 'rsrq','protocole']]
#Keep only DOWN protocol as we are only interested in download speed
qos_df =qos_df[qos_df["protocole"] == "DOWNLOAD"]

# delete empty cells coming from the absence of rsrp, rsrq values
qos_df = qos_df.dropna(subset=["rsrp", "rsrp"]).reset_index()        

#1.2 import geo_po file and merge it with qos
population_df = pd.read_csv(project_root/config["data"]["clean"]["file1"], sep=(","), dtype={"insee_com":"str"}, encoding = "latin1")
qos_po_df = qos_df.merge(population_df[["insee_com","population"]] , on="insee_com", how="left").reset_index(drop=True)

#1.3 merge qos_po with sites
sites_df = pd.read_csv(project_root/config["data"]["clean"]["file2"], sep=(","), dtype={"insee_com":"str"}, encoding = "latin1")
sites_per_commune = sites_df.groupby(["insee_com", "nom_op"]).agg(nbr_phys_sites=("site_id", "count"), nbr_pure_5g_sites=("site_5g_3500_m_hz", "sum")).reset_index()
sites_per_commune["operator"] = sites_per_commune["nom_op"].map({"Bouygues Telecom": "Bouygues", "Free Mobile":"Free", "Orange":"Orange", "SFR":"SFR"})

qos_po_sites_df =qos_po_df.merge(sites_per_commune, on=["insee_com", "operator"], how="left").reset_index(drop=True)

#Our ml dataset is:
ml_df = (
    qos_po_sites_df.groupby(["insee_com", "operator"])
      .agg(
          download_speed=("bitrate_dl", "median"),
          rsrp=("rsrp", "median"),
          rsrq=("rsrq", "median"),
          population=("population", "first"),
          tot_phys_sites=("nbr_phys_sites", "first"),
          tot_5g_per_op_com=("nbr_pure_5g_sites", "first")
      )
      .reset_index()
)
#Delete empty cells coming from empty sites
ml_df = ml_df.dropna(subset="tot_phys_sites").reset_index()
ml_df.head()

,index,insee_com,operator,download_speed,rsrp,rsrq,population,tot_phys_sites,tot_5g_per_op_com
0,0,01043,Bouygues,307.179138,-102.360,-13.285,5198.0,3.0,2.0
1,1,01043,Free,130.842855,-108.250,-14.800,5198.0,2.0,1.0
2,2,01043,Orange,197.726610,-112.100,-14.465,5198.0,2.0,1.0
3,3,01043,SFR,106.592793,-109.945,-15.185,5198.0,2.0,2.0
4,4,01071,Bouygues,233.677361,-100.475,-9.990,5832.0,1.0,0.0


In [111]:
# 2. Define features and target columns
X = ml_df[
    [
     "operator",
        "population",
        "tot_phys_sites", 
        "tot_5g_per_op_com",
        "rsrp",
        "rsrq"
    ]
].copy()

y = ml_df["download_speed"].copy()

#3 Train, test split  by commune
groups = ml_df["insee_com"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("Training communes:", ml_df.iloc[train_idx]["insee_com"].nunique())
print("Test communes:", ml_df.iloc[test_idx]["insee_com"].nunique())


Training set: (4267, 6)
Test set: (1084, 6)
Training communes: 1188
Test communes: 297


In [112]:
# ----------------------------------------------------------------------------
# I. TRAIN DECISION TREE MODEL
# ----------------------------------------------------------------------------
#4. preprocessing: 
categorical_features = ["operator"]
numeric_features = ["population","tot_5g_per_op_com", "tot_phys_sites","rsrp","rsrq"] 

encoder = OneHotEncoder(sparse_output=False)
#fit X_train only
X_operator_encoded = encoder.fit(X_train[["operator"]])

#Transform data: Xtrain+X_test
operator_train_encoded = encoder.transform(X_train[["operator"]])
operator_test_encoded = encoder.transform(X_test[["operator"]])

#Get the numerical variables
X_train_numeric = X_train[numeric_features].values
X_test_numeric = X_test[numeric_features].values
                                          
#Combine them
X_train_final = np.hstack([
    operator_train_encoded,
    X_train_numeric
])

X_test_final = np.hstack([
    operator_test_encoded,
    X_test_numeric
])
print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)

#5 define the Decision tree model and fit it:
dt_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=10,
    min_samples_leaf=5
)

dt_model.fit(X_train_final, y_train)

# 6. Predict the test set
y_pred_dt = dt_model.predict(X_test_final)

#Evaluate the model
r2_dt = r2_score(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print(f"R²   : {r2_dt:.3f}")
print(f"MAE  : {mae_dt:.2f} Mbps")
print(f"RMSE : {rmse_dt:.2f} Mbps")

X_train_final: (4267, 9)
X_test_final: (1084, 9)
R²   : 0.478
MAE  : 81.11 Mbps
RMSE : 113.32 Mbps


In [105]:
# ----------------------------------------------------------------------------
# II. TRAIN LINEAR REGRESSION MODEL
# ----------------------------------------------------------------------------
# 1. Preprocessing:
encoder_lr = OneHotEncoder(sparse_output=False)
#fit Train dataset
encoder_lr.fit(X_train[["operator"]])

#transforù both Xtrain and Xtest
operator_train_encoded_lr = encoder.transform(X_train[["operator"]])
operator_test_encoded_lr = encoder_lr.transform(X_test[["operator"]])

# 2. standardize the numerical features
scaler = StandardScaler()
X_train_numeric_scaled = scaler.fit_transform(
    X_train[numeric_features]
)
X_test_numeric_scaled = scaler.transform(
    X_test[numeric_features]
)
                                              
# 3. Combine categorical + numerical variables
X_train_lr = np.hstack([
    operator_train_encoded_lr,
    X_train_numeric_scaled
])

X_test_lr = np.hstack([
    operator_test_encoded_lr,
    X_test_numeric_scaled
])

#4. Train model
lr_model = LinearRegression()

lr_model.fit(X_train_lr, y_train)

#5. Evaluate model:
y_pred_lr = lr_model.predict(X_test_lr)

r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print(f"R²   : {r2_lr:.3f}")
print(f"MAE  : {mae_lr:.2f} Mbps")
print(f"RMSE : {rmse_lr:.2f} Mbps")

(4267, 9)
(1084, 9)
R²   : 0.385
MAE  : 99.33 Mbps
RMSE : 123.04 Mbps


In [ ]:
# initiate model: knn regressor
knn = KNeighborsRegressor(n_neighbors=10)
knn.fit(X_train, y_train)

print(f"The R2 of the model is {knn.score(X_test, y_test): .2f}")

In [73]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (5351, 6)
y shape: (5351,)

Features:
['operator', 'population', 'tot_phys_sites', 'tot_5g_per_op_com', 'rsrp', 'rsrq']


In [93]:
print(encoder.categories_)
print(X_train_final.shape)
print(X_test_final.shape)

[array(['Bouygues', 'Free', 'Orange', 'SFR'], dtype=object)]
(4267, 9)
(1084, 9)


In [87]:
train_communes = set(ml_df.iloc[train_idx]["insee_com"])
test_communes = set(ml_df.iloc[test_idx]["insee_com"])

print("Common communes:", len(train_communes & test_communes))

Common communes: 0
